In [1]:
#%matplotlib inline
%time from hikyuu.interactive import *

Initialize hikyuu_2.8.2_202609250345_RELEASE_macosx_arm64 ...
2026-09-25 13:38:23.182 [HKU-I] - current python version: 3.13.12 (main.cpp:77)


2026-09-25 13:38:24,594 [INFO] runing in interactive session [<module>] (/Users/fasiondog/workspace/hikyuu/hikyuu/__init__.py:149) [hikyuu::hku_info]
2026-09-25 13:38:24,595 [INFO] running in jupyter [<module>] (/Users/fasiondog/workspace/hikyuu/hikyuu/__init__.py:156) [hikyuu::hku_info]


2026-09-25 13:38:24.599 [HKU-I] - 插件路径: /Users/fasiondog/workspace/hku_plugin/hikyuu_plugin (StockManager.cpp:124)
2026-09-25 13:38:24.630 [HKU-I] - Using CLICKHOUSE BaseInfoDriver (BaseInfoDriver.cpp:56)
2026-09-25 13:38:24.653 [HKU-I] - 加载市场信息…… (StockManager.cpp:1050)
2026-09-25 13:38:24.664 [HKU-I] - 加载证券类型信息…… (StockManager.cpp:1067)
2026-09-25 13:38:24.674 [HKU-I] - 加载证券信息…… (StockManager.cpp:948)
2026-09-25 13:38:24.826 [HKU-I] - 加载权息数据…… (StockManager.cpp:1170)
2026-09-25 13:38:25.118 [HKU-I] - 加载板块信息…… (StockManager.cpp:214)
2026-09-25 13:38:25.252 [HKU-I] - 加载K线数据…… (StockManager.cpp:221)
2026-09-25 13:38:25.252 [HKU-I] - 预加载 day K线数据至缓存 (最大数量: 100000)! (StockManager.cpp:395)
2026-09-25 13:38:25.252 [HKU-I] - 0.61 秒数据加载完毕. (StockManager.cpp:231)
CPU times: user 2.34 s, sys: 209 ms, total: 2.55 s
Wall time: 3.33 s


# 1 利用 TM 实现简单的记账本

TradeManager对象可以理解为一个模拟的交易账户，负责交易的买/卖操作、记录交易记录以及持仓情况，也可以通过修改其买/卖操作的接口实现实盘接入。创建一个模拟交易账户，通常使用快捷创建函数 crtTM。TM对象的基本操作：

- buy  买入
- sell 卖出
- checkin 存入现金
- checkout 取出现金

可以利用 TM 实现简单的记账本，手工记录自己的操作情况，例如：

In [2]:
# Create a simulated account with an initial capital of 100,000, starting from 2017-01-01
my_tm = crtTM(init_cash=100000, date=Datetime(201701010000))

# On 2017-01-03, buy 100 shares at the price of 9.11
td = my_tm.buy(Datetime(201701030000), sm['sz000001'], 9.11, 100)

# View the current cash and position
print(my_tm)

TradeManager {
  params: params[precision(int): 2, save_action(bool): 1, support_borrow_cash(bool): 0, support_borrow_stock(bool): 0, ],
  name: SYS,
  init_date: 2017-01-01 00:00:00,
  init_cash: 100000.00,
  firstDatetime: 2017-01-03 00:00:00,
  lastDatetime: 2017-01-03 00:00:00,
  TradeCostFunc: TradeCostFunc(TC_Zero, params[]),
  current total funds: 100005.00,
  current cash: 99089.00,
  current market_value: 916.00,
  current short_market_value: 0.00,
  current base_cash: 100000.00,
  current base_asset: 0.00,
  current borrow_cash: 0.00,
  current borrow_asset: 0.00,
  Position: 
    证券代码 证券名称 建仓日期 持有天数 持有数量 投入资金 当前市值 当前收益 当前收益率 相对账户初始资金收益率
    SZ000001 平安银行 2017-01-03 00:00:00 2364 100.00 911.00 1130.00 219.00 24.04% 0.22%
  Short Position: 
  Borrow Stock: 
}


In [3]:
# Convert to a pandas DataFrame to display the current positions 
position = my_tm.get_position_list()
position.to_df()

,证券代码,证券名称,开仓时间,持有天数,持仓数量,总投入资金,市值,收益,收益百分比,止损,目标价格,清仓时间,累计持仓数量,累计成本,累计风险,累计买入资金,累计卖出资金
0,SZ000001,平安银行,2017-01-03,3552,100.0,911.0,1130.0,219.0,24.04,0.0,0.0,NaT,100.0,0.0,911.0,911.0,0.0


In [4]:
my_tm.get_trade_list().to_df()

,证券代码,证券名称,日期,交易操作,计划价格,实际价格,目标价格,数量,止损,现金,总成本,佣金,印花税,过户费,其它成本,来源,备注
0,,,2017-01-01,INIT,100000.0,100000.00,0.0,0.0,0.0,100000.0,0.0,0.0,0.0,0.0,0.0,--,
1,SZ000001,平安银行,2017-01-03,BUY,0.0,9.11,0.0,100.0,0.0,99089.0,0.0,0.0,0.0,0.0,0.0,--,


In [5]:
# On 2017-02-21, sell 100 shares at the price of 9.60
td = my_tm.sell(Datetime(201702210000), sm['sz000001'], 9.60)

my_tm

TradeManager {
  params: params[precision(int): 2, save_action(bool): 1, support_borrow_cash(bool): 0, support_borrow_stock(bool): 0, ],
  name: SYS,
  init_date: 2017-01-01 00:00:00,
  init_cash: 100000.00,
  firstDatetime: 2017-01-03 00:00:00,
  lastDatetime: 2017-02-21 00:00:00,
  TradeCostFunc: TradeCostFunc(TC_Zero, params[]),
  current total funds: 100049.00,
  current cash: 100049.00,
  current market_value: 0.00,
  current short_market_value: 0.00,
  current base_cash: 100000.00,
  current base_asset: 0.00,
  current borrow_cash: 0.00,
  current borrow_asset: 0.00,
  Position: 
    证券代码 证券名称 建仓日期 持有天数 持有数量 投入资金 当前市值 当前收益 当前收益率 相对账户初始资金收益率
  Short Position: 
  Borrow Stock: 
}

# 2 利用 Excel 查看交易详情

使用 tocsv 方法将 TM 的交易记录、当前持仓及已平仓详细情况分别保存为 csv 文件，以便用 Excel 查看详情。

tocsv方法参数为一个指定的目录，目录必须以存在。其输出会在指定目录中，生成三个文件，“TM名称_交易记录.csv”、“TM名称_未平仓记录.csv”、“TM名称_已平仓记录.csv”。TM名称可在crtTM创建TM对象时指定，默认为“SYS”，如下图所示。

<img src="images/008_01_tocsv.png" align='left'>

In [6]:
# Output to the temporary path configured in the hikyuu_XXX.ini file
my_tm.tocsv(sm.tmpdir())

使用 Excel 查看 csv，如：

<img src="images/008_02_tocsv_look.png" align="left">

# 3 使用序列化保存或重新载入已有TM对象

In [7]:
# Save to the specified file
from datetime import date
filename = "my_trade_record_{}.pkl".format(date.today())
hku_save(my_tm, filename)

In [8]:
# Load the saved TM object
new_my_tm = hku_load(filename)

# 4 使用订单代理

In [9]:
# Create a simulated trading account for backtesting with an initial capital of 300,000
my_tm = crtTM(init_cash=300000, date=Datetime(201701010000))

# Register the live trading order broker
ob = crtOB(TestOrderBroker())
my_tm.reg_broker(ob) # TestOerderBroker is a test order broker object that only prints
# Note: pybind does not support the following calling style; you must create the instance first and then pass it!!!
# my_tm.reg_broker(crtOB(TestOrderBroker(), False))

# Modify the last datetime of the order broker as needed; only after this datetime will the order broker actually issue the order instructions
my_tm.broker_last_datetime=Datetime(201701010000)

# Create the signal generator (the 5-day EMA as the fast line and the 10-day EMA of the 5-day EMA itself as the slow line; buy when the fast line crosses the slow line upward, and sell otherwise)
my_sg = SG_Flex(EMA(C, n=5), slow_n=10)

# Fixedly buy 1000 shares each time
my_mm = MM_FixedCount(1000)

# Create the trading system and run it
sys = SYS_Simple(tm = my_tm, sg = my_sg, mm = my_mm)
sys.run(sm['sz000001'], Query(-150))

Buy: SZ000001, price: 10.93, number: 1000.0, expected stop-loss price: 0.0, expected goal price: nan, signal source: SystemPart.SIGNAL, remark: 
Sell: SZ000001, price: 10.68, number: 1000.0, signal source: SystemPart.SIGNAL, remark: 
Buy: SZ000001, price: 10.909, number: 1000.0, expected stop-loss price: 0.0, expected goal price: nan, signal source: SystemPart.SIGNAL, remark: 
Sell: SZ000001, price: 11.01, number: 1000.0, signal source: SystemPart.SIGNAL, remark: 
Buy: SZ000001, price: 11.359, number: 1000.0, expected stop-loss price: 0.0, expected goal price: nan, signal source: SystemPart.SIGNAL, remark: 
Sell: SZ000001, price: 11.14, number: 1000.0, signal source: SystemPart.SIGNAL, remark: 
Buy: SZ000001, price: 10.979, number: 1000.0, expected stop-loss price: 0.0, expected goal price: nan, signal source: SystemPart.SIGNAL, remark: 
Sell: SZ000001, price: 10.74, number: 1000.0, signal source: SystemPart.SIGNAL, remark: 
Buy: SZ000001, price: 10.55, number: 1000.0, expected stop-lo

In [10]:
my_tm.get_trade_list().to_df()

,证券代码,证券名称,日期,交易操作,计划价格,实际价格,目标价格,数量,止损,现金,总成本,佣金,印花税,过户费,其它成本,来源,备注
0,,,2017-01-01,INIT,300000.000,300000.000,0.0,0.0,0.0,300000.0,0.0,0.0,0.0,0.0,0.0,--,
1,SZ000001,平安银行,2026-03-13,BUY,10.930,10.930,NaN,1000.0,0.0,289070.0,0.0,0.0,0.0,0.0,0.0,SG,
2,SZ000001,平安银行,2026-03-23,SELL,10.680,10.680,NaN,1000.0,0.0,299750.0,0.0,0.0,0.0,0.0,0.0,SG,
3,SZ000001,平安银行,2026-03-27,BUY,10.909,10.909,NaN,1000.0,0.0,288841.0,0.0,0.0,0.0,0.0,0.0,SG,
4,SZ000001,平安银行,2026-04-20,SELL,11.010,11.010,NaN,1000.0,0.0,299851.0,0.0,0.0,0.0,0.0,0.0,SG,
5,SZ000001,平安银行,2026-04-28,BUY,11.359,11.359,NaN,1000.0,0.0,288492.0,0.0,0.0,0.0,0.0,0.0,SG,
6,SZ000001,平安银行,2026-05-14,SELL,11.140,11.140,NaN,1000.0,0.0,299632.0,0.0,0.0,0.0,0.0,0.0,SG,
7,SZ000001,平安银行,2026-06-02,BUY,10.979,10.979,NaN,1000.0,0.0,288653.0,0.0,0.0,0.0,0.0,0.0,SG,
8,SZ000001,平安银行,2026-06-12,BONUS,360.000,360.000,0.0,0.0,0.0,289373.0,0.0,0.0,0.0,0.0,0.0,--,
9,SZ000001,平安银行,2026-06-18,SELL,10.740,10.740,NaN,1000.0,0.0,299753.0,0.0,0.0,0.0,0.0,0.0,SG,


In [11]:
my_tm.get_trade_list().to_np()

array([('', '', '2017-01-01T00:00:00.000000000', 'INIT', 3.0000e+05, 3.0000e+05,  0.,    0., 0., 300000., 0., 0., 0., 0., 0., '--', ''),
       ('SZ000001', '平安银行', '2026-03-13T00:00:00.000000000', 'BUY', 1.0930e+01, 1.0930e+01, nan, 1000., 0., 289070., 0., 0., 0., 0., 0., 'SG', ''),
       ('SZ000001', '平安银行', '2026-03-23T00:00:00.000000000', 'SELL', 1.0680e+01, 1.0680e+01, nan, 1000., 0., 299750., 0., 0., 0., 0., 0., 'SG', ''),
       ('SZ000001', '平安银行', '2026-03-27T00:00:00.000000000', 'BUY', 1.0909e+01, 1.0909e+01, nan, 1000., 0., 288841., 0., 0., 0., 0., 0., 'SG', ''),
       ('SZ000001', '平安银行', '2026-04-20T00:00:00.000000000', 'SELL', 1.1010e+01, 1.1010e+01, nan, 1000., 0., 299851., 0., 0., 0., 0., 0., 'SG', ''),
       ('SZ000001', '平安银行', '2026-04-28T00:00:00.000000000', 'BUY', 1.1359e+01, 1.1359e+01, nan, 1000., 0., 288492., 0., 0., 0., 0., 0., 'SG', ''),
       ('SZ000001', '平安银行', '2026-05-14T00:00:00.000000000', 'SELL', 1.1140e+01, 1.1140e+01, nan, 1000., 0., 299632., 0.,

In [12]:
my_tm.get_history_position_list().to_df()

,证券代码,证券名称,开仓时间,持有天数,持仓数量,总投入资金,市值,收益,收益百分比,止损,目标价格,清仓时间,累计持仓数量,累计成本,累计风险,累计买入资金,累计卖出资金
0,SZ000001,平安银行,2026-03-13,10,0.0,250.0,0.0,-250.0,-100.0,0.0,NaN,2026-03-23,1000.0,0.0,10930.0,10930.0,10680.0
1,SZ000001,平安银行,2026-03-27,24,0.0,-101.0,0.0,101.0,-100.0,0.0,NaN,2026-04-20,1000.0,0.0,10909.0,10909.0,11010.0
2,SZ000001,平安银行,2026-04-28,16,0.0,219.0,0.0,-219.0,-100.0,0.0,NaN,2026-05-14,1000.0,0.0,11359.0,11359.0,11140.0
3,SZ000001,平安银行,2026-06-02,16,0.0,-121.0,0.0,121.0,-100.0,0.0,NaN,2026-06-18,1000.0,0.0,10979.0,10979.0,11100.0
4,SZ000001,平安银行,2026-07-09,35,0.0,-680.0,0.0,680.0,-100.0,0.0,NaN,2026-08-13,1000.0,0.0,10550.0,10550.0,11230.0
5,SZ000001,平安银行,2026-08-21,28,0.0,-230.0,0.0,230.0,-100.0,0.0,NaN,2026-09-18,1000.0,0.0,11360.0,11360.0,11590.0


In [13]:
my_tm.get_history_position_list().to_np()

array([('SZ000001', '平安银行', '2026-03-13T00:00:00.000000000', 10, 0.,  250., 0., -250., -100., 0., nan, '2026-03-23T00:00:00.000000000', 1000., 0., 10930., 10930., 10680.),
       ('SZ000001', '平安银行', '2026-03-27T00:00:00.000000000', 24, 0., -101., 0.,  101., -100., 0., nan, '2026-04-20T00:00:00.000000000', 1000., 0., 10909., 10909., 11010.),
       ('SZ000001', '平安银行', '2026-04-28T00:00:00.000000000', 16, 0.,  219., 0., -219., -100., 0., nan, '2026-05-14T00:00:00.000000000', 1000., 0., 11359., 11359., 11140.),
       ('SZ000001', '平安银行', '2026-06-02T00:00:00.000000000', 16, 0., -121., 0.,  121., -100., 0., nan, '2026-06-18T00:00:00.000000000', 1000., 0., 10979., 10979., 11100.),
       ('SZ000001', '平安银行', '2026-07-09T00:00:00.000000000', 35, 0., -680., 0.,  680., -100., 0., nan, '2026-08-13T00:00:00.000000000', 1000., 0., 10550., 10550., 11230.),
       ('SZ000001', '平安银行', '2026-08-21T00:00:00.000000000', 28, 0., -230., 0.,  230., -100., 0., nan, '2026-09-18T00:00:00.000000000', 1000